# 04 - Model preparation
Turn the feature file into a numbers-only file that the models can use. Also save what the Streamlit app will need later.

## Step 1: Load the data

In [1]:
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/campaigns_features.csv")
print(df.shape)

(166665, 26)


## Step 2: Save the dropdown options for the Streamlit app
We do this before encoding, while the text columns are still text.

In [2]:
dropdown_options = {
    "Brand": sorted(df["Brand"].unique()),
    "Campaign_Type": sorted(df["Campaign_Type"].unique()),
    "Target_Audience": sorted(df["Target_Audience"].unique()),
    "Language": sorted(df["Language"].unique()),
    "Customer_Segment": sorted(df["Customer_Segment"].unique()),
}

joblib.dump(dropdown_options, "../models/dropdown_options.pkl")
dropdown_options

{'Brand': ['Nykaa', 'Purplle', 'Tira'],
 'Campaign_Type': ['Email',
  'Influencer',
  'Paid Ads',
  'SEO',
  'Social Media',
  'Unknown'],
 'Target_Audience': ['College Students',
  'Premium Shoppers',
  'Tier 2 City Customers',
  'Unknown',
  'Working Women',
  'Youth'],
 'Language': ['Bengali', 'English', 'Hindi', 'Tamil', 'Unknown'],
 'Customer_Segment': ['College Students',
  'Premium Shoppers',
  'Tier 2 City Customers',
  'Unknown',
  'Working Women',
  'Youth']}

## Step 3: Drop columns that must not be inputs
- `Campaign_ID` is only a label.
- `Channel_Used` is replaced by the 6 channel 0/1 columns.

In [3]:
df = df.drop(columns=["Campaign_ID", "Channel_Used"])

## Step 4: Turn the 5 text columns into 0/1 columns

In [4]:
text_columns = ["Brand", "Campaign_Type", "Target_Audience", "Language", "Customer_Segment"]
df = pd.get_dummies(df, columns=text_columns, drop_first=True, dtype=int)
print(df.shape)

(166665, 40)


## Step 5: Choose the inputs (X)
`Revenue` and `Profit_Flag` are the targets. `ROI` is calculated from Revenue and cost, so it would leak the answer. All three stay out of X.

In [5]:
X = df.drop(columns=["Revenue", "ROI", "Profit_Flag"])

print("Number of input columns:", X.shape[1])
print("Any non-numeric columns?", (X.dtypes == "object").sum())
X.dtypes

Number of input columns: 37
Any non-numeric columns? 0


Duration                                    int64
Impressions                                 int64
Clicks                                      int64
Leads                                       int64
Conversions                                 int64
Acquisition_Cost                          float64
Engagement_Score                          float64
Year                                        int64
Month                                       int64
Channel_Instagram                           int64
Channel_Email                               int64
Channel_Google                              int64
Channel_Facebook                            int64
Channel_YouTube                             int64
Channel_WhatsApp                            int64
Channel_Count                               int64
Brand_Purplle                               int64
Brand_Tira                                  int64
Campaign_Type_Influencer                    int64
Campaign_Type_Paid Ads                      int64


## Step 6: Save the model-ready file and the column list

In [6]:
model_ready = X.copy()
model_ready["Revenue"] = df["Revenue"]
model_ready["Profit_Flag"] = df["Profit_Flag"]

model_ready.to_csv("../data/processed/campaigns_model_ready.csv", index=False)
joblib.dump(list(X.columns), "../models/model_columns.pkl")

print(model_ready.shape)
print("Saved!")

(166665, 39)
Saved!
